# 科研论文助手（Scientific Paper Agent）：用 LangGraph 做检索、阅读与综述

## 概述

这个项目实现了一个智能研究助理：它使用 LangGraph 与大语言模型来帮助用户检索、阅读、理解与分析科研文献。通过结合学术检索 API 与论文处理（例如 PDF 文本抽取），系统可以为研究者、学生与工程实践者提供更顺滑的文献调研体验。

> 提示：这个工作流并不限定在“论文”领域，只要更换提示词与工具，同样的结构也可以迁移到其他领域。

## 动机

文献调研往往占据大量研发时间。有研究指出，研究人员可能会把 30–50% 的时间投入到阅读、分析与综合学术论文上。这是研究社区普遍存在的挑战。系统化的文献综述对于推动科研与工程创新很关键，但传统流程仍然低效、耗时。

常见痛点包括：
- 大量时间成本（30–50% 的研发工时）消耗在阅读与整理上
- 数据库/平台分散，检索入口碎片化
- 跨多篇论文整合观点与证据的工作复杂
- 维护一份高质量、可更新的综述成本高
- 持续跟进新论文需要长期投入

## 关键组件

1) **State 驱动的工作流引擎**
- StateGraph 架构：用多个节点编排“调研”流程
- 决策节点：分析问题意图并做路由
- 规划节点：生成研究计划与检索策略
- 工具执行节点：检索论文、下载与处理文档
- 评审节点：质量校验与必要的迭代改进

2) **论文处理与数据源整合**
- 数据源接入：使用 CORE API 做论文检索
- 文档处理：下载 PDF 并抽取文本，尽量保留结构

3) **分析工作流**
- 基于 state 的多步流水线
- 多重校验与门控
- 以质量为导向的改进循环
- 可选的人类介入

工作流示意图：

![image](https://i.ibb.co/0BBzkcb/mermaid-diagram-2024-11-17-195744.png)

## 方法细节

1) **运行所需的 Key**
- **DashScope API Key**：用于调用大语言模型（本 notebook 通过 `DASHSCOPE_API_KEY` 与 `DASHSCOPE_BASE_URL` 初始化 `ChatOpenAI`）
- **CORE API Key**：用于论文检索。CORE 是一个大型开放论文库（规模达上亿篇），提供个人免费 API Key。可以在 [这里](https://core.ac.uk/services/api#form) 申请。

2) **技术架构**
- LangGraph：负责 state 编排与节点路由
- pdfplumber：负责 PDF 文本抽取
- Pydantic：负责结构化输入/输出与校验

> 致谢：感谢 CORE API 提供学术论文访问能力。

---

## 环境准备

这一节会导入所需依赖并加载环境变量。

In [ ]:
# 请确保已安装以下依赖：
# - langchain
# - langchain-openai
# - langgraph
# - pdfplumber
# - python-dotenv
# - urllib3
# - pydantic

这一节导入所需库，并从 `../.env` 加载环境变量。

In [ ]:
import io
import json
import os
import urllib3
import time

import pdfplumber
from dotenv import load_dotenv
from IPython.display import display, Markdown
from langchain_core.messages import BaseMessage, SystemMessage, ToolMessage, AIMessage
from langchain_core.tools import BaseTool, tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from typing import Annotated, ClassVar, Sequence, TypedDict, Optional

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# 在.env文件中设置自己的秘钥，包括CORE_API_KEY
load_dotenv("../.env")

llm = ChatOpenAI(
    model="deepseek-v4-flash-0731",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    temperature=0.0,
)

# You can set your own keys here
# os.environ["OPENAI_API_KEY"] = "sk-proj-..."
# os.environ["CORE_API_KEY"] = "..."

## 提示词

这一节定义工作流里用到的提示词。

其中 `agent_prompt` 包含 CORE API 的查询语法说明，用于支持更复杂的检索需求（例如按年份/字段过滤）。

In [ ]:
# 决策：判断是否需要检索证据
decision_making_prompt = """
你是一名经验丰富的科研助理。
你的目标是帮助用户完成科研调研与文献综述相关任务。

根据用户问题，判断是否需要检索证据（research），还是可以直接回答。
- 需要检索证据：用户的问题需要论文/资料支撑、需要引用来源、需要对多篇文献做综合或对比。
- 可以直接回答：非常简单的闲聊或不依赖外部证据的问题（例如“你好吗？”）。
""".strip()


# 规划：把问题拆成可执行步骤，并标注每步使用的工具
planning_prompt = """
# 角色
你是一名经验丰富的科研助理。

# 任务
制定一个新的、可执行的逐步计划，帮助用户完成科研调研任务。

# 约束
- 子任务不要凭空猜测；缺信息就通过工具检索或下载原文。
- 如果对话里出现对上一版答案的反馈，需要把反馈融入新计划。

# 工具
对每个子任务，明确需要使用的外部工具。可用工具如下：
{tools}
""".strip()


# 执行：按计划完成调研，并在答案中给出引用
agent_prompt = """
# 角色
你是一名经验丰富的科研助理。
你的目标是帮助用户完成科研调研。你可以调用外部工具来检索/下载论文并阅读。

# 行为准则
- 按你写的计划执行。
- 对关键结论给出充分的“句内引用”（inline citations）。

# 外部知识：CORE API 查询语法

CORE API 支持一种查询语言，用于执行更复杂的检索。常见操作符如下：

| 操作符 | 可用符号 | 含义 |
|---|---|---|
| And | AND, +, 空格 | 逻辑与 |
| Or | OR | 逻辑或 |
| Grouping | (...) | 分组与优先级 |
| Field lookup | field_name:value | 按字段查询 |
| Range queries | fieldName(>, <,>=, <=) | 数值/日期范围查询 |
| Exists queries | _exists_:fieldName | 字段存在性过滤（字段非空） |

你可以用它来构造更复杂的检索（例如限定年份/作者/标题）。下面是论文对象中常用字段示例：
{
  "authors": [{"name": "Last Name, First Name"}],
  "documentType": "presentation" or "research" or "thesis",
  "publishedDate": "2019-08-24T14:15:22Z",
  "title": "Title of the paper",
  "yearPublished": "2019"
}

示例查询：
- "machine learning AND yearPublished:2023"
- "maritime biology AND yearPublished>=2023 AND yearPublished<=2024"
- "cancer research AND authors:Vaswani, Ashish AND authors:Bello, Irwan"
- "title:Attention is all you need"
- "mathematics AND _exists_:abstract"
""".strip()


# 评审：判断最终答案是否合格，不合格则给出可执行反馈
judge_prompt = """
你是一名严格的科研评审。
你的目标是评估针对用户问题的最终回答是否合格。

你需要阅读对话历史，并判断：最终回答是否满足用户需求。

合格回答应当：
- 直接回答用户问题
- 回答足够充分、具体
- 关键结论包含句内引用，以支持每个关键主张

如果不合格，请给出清晰、可执行的改进反馈（feedback），指出需要补充/修正的点。
""".strip()

## 工具封装与通用函数

这一节包含工作流中用到的工具封装与通用函数：

- CORE API 的简单封装（包含重试机制）
- 节点输入/输出的 Pydantic 模型
- `print_stream`：用于把工作流的执行过程以流式方式打印出来

In [ ]:
class CoreAPIWrapper(BaseModel):
    """CORE API 的简单封装。"""

    base_url: ClassVar[str] = "https://api.core.ac.uk/v3"
    api_key: ClassVar[str] = os.environ.get("CORE_API_KEY") or ""

    top_k_results: int = Field(description="返回的结果条数", default=1)

    def _get_search_response(self, query: str) -> dict:
        if not self.api_key:
            raise ValueError("Missing CORE_API_KEY")

        http = urllib3.PoolManager()

        max_retries = 5
        for attempt in range(max_retries):
            response = http.request(
                "GET",
                f"{self.base_url}/search/outputs",
                headers={"Authorization": f"Bearer {self.api_key}"},
                fields={"q": query, "limit": self.top_k_results},
            )
            if 200 <= response.status < 300:
                return json.loads(response.data)
            if attempt < max_retries - 1:
                time.sleep(2 ** (attempt + 2))
            else:
                raise Exception(f"Got non-2xx response from CORE API: {response.status} {response.data}")

        raise RuntimeError("unreachable")

    def search(self, query: str) -> str:
        response = self._get_search_response(query)
        results = response.get("results", [])
        if not results:
            return "No relevant results were found"

        docs = []
        for result in results:
            published_date_str = result.get("publishedDate") or result.get("yearPublished", "")
            authors_str = " and ".join([item.get("name", "") for item in result.get("authors", [])])
            docs.append(
                (
                    f"* ID: {result.get('id', '')},\n"
                    f"* Title: {result.get('title', '')},\n"
                    f"* Published Date: {published_date_str},\n"
                    f"* Authors: {authors_str},\n"
                    f"* Abstract: {result.get('abstract', '')},\n"
                    f"* Paper URLs: {result.get('sourceFulltextUrls') or result.get('downloadUrl', '')}"
                )
            )
        return "\n-----\n".join(docs)


class SearchPapersInput(BaseModel):
    """用 CORE API 搜索论文。"""

    query: str = Field(description="检索查询")
    max_papers: int = Field(description="返回的最大论文数量（1-10）", default=1, ge=1, le=10)


class DecisionMakingOutput(BaseModel):
    """决策节点输出。"""

    requires_research: bool = Field(description="是否需要检索证据")
    answer: Optional[str] = Field(default=None, description="不需要检索时可直接给出答案")


class JudgeOutput(BaseModel):
    """评审节点输出。"""

    is_good_answer: bool = Field(description="答案是否合格")
    feedback: Optional[str] = Field(default=None, description="不合格时的改进反馈")


def format_tools_description(tools: list[BaseTool]) -> str:
    return "\n\n".join([f"- {t.name}: {t.description}\n  Input arguments: {t.args}" for t in tools])


async def print_stream(app: CompiledStateGraph, input: str) -> Optional[BaseMessage]:
    display(Markdown("## 新的调研任务"))
    display(Markdown(f"### 输入\n\n{input}\n\n"))
    display(Markdown("### 流式输出\n\n"))

    init_state = {
        "requires_research": False,
        "num_feedback_requests": 0,
        "is_good_answer": False,
        "messages": [{"role": "user", "content": input}],
    }

    all_messages: list[BaseMessage] = []
    async for chunk in app.astream(init_state, stream_mode="updates"):
        for updates in chunk.values():
            if messages := updates.get("messages"):
                all_messages.extend(messages)
                for message in messages:
                    message.pretty_print()
                    print("\n\n")

    if not all_messages:
        return None
    return all_messages[-1]

## State 定义

这一节定义工作流的 state，主要包含：
- `requires_research`：是否需要检索证据
- `num_feedback_requests`：自检反馈的轮数
- `is_good_answer`：最终答案是否合格
- `messages`：对话历史（包含 AIMessage 与 ToolMessage）

In [ ]:
class AgentState(TypedDict):
    """The state of the agent during the paper research process."""
    requires_research: bool = False
    num_feedback_requests: int = 0
    is_good_answer: bool = False
    messages: Annotated[Sequence[BaseMessage], add_messages]

## 工具（Tools）

这一节定义 Agent 可用的工具：

- `search-papers`：通过 CORE API 搜索论文
- `download-paper`：从给定 URL 下载论文并抽取 PDF 文本
- `ask-human-feedback`：在需要时向人类提问获取反馈

为了提高下载成功率，`download-paper` 会携带浏览器请求头，并包含简单的重试逻辑。

In [ ]:
@tool("search-papers", args_schema=SearchPapersInput)
def search_papers(query: str, max_papers: int = 1) -> str:
    """通过 CORE API 搜索论文。

    Example:
    {"query": "Attention is all you need", "max_papers": 1}

    Returns:
        返回匹配论文的列表（包含标题、作者、摘要、URL 等关键信息）。
    """
    try:
        return CoreAPIWrapper(top_k_results=max_papers).search(query)
    except Exception as e:
        return f"Error performing paper search: {e}"


@tool("download-paper")
def download_paper(url: str) -> str:
    """从 URL 下载 PDF 并抽取文本。"""

    try:
        http = urllib3.PoolManager(cert_reqs="CERT_NONE")

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Accept-Encoding": "gzip, deflate, br",
            "Connection": "keep-alive",
        }

        max_retries = 5
        for attempt in range(max_retries):
            response = http.request("GET", url, headers=headers)
            if 200 <= response.status < 300:
                pdf_file = io.BytesIO(response.data)
                with pdfplumber.open(pdf_file) as pdf:
                    text = ""
                    for page in pdf.pages:
                        text += (page.extract_text() or "") + "\n"
                return text
            if attempt < max_retries - 1:
                time.sleep(2 ** (attempt + 2))
            else:
                raise Exception(f"Got non-2xx when downloading paper: {response.status} {response.data}")

        raise RuntimeError("unreachable")
    except Exception as e:
        return f"Error downloading paper: {e}"


@tool("ask-human-feedback")
def ask_human_feedback(question: str) -> str:
    """向人类提问获取反馈（遇到不确定时使用）。"""

    return input(question)


tools = [search_papers, download_paper, ask_human_feedback]
tools_dict = {t.name: t for t in tools}

## 工作流节点

这一节定义工作流的各个节点。

注意：`judge_node` 被设置为最多反馈 2 次就结束，以控制整体延迟。

In [ ]:
# LLMs
base_llm = llm
decision_making_llm = base_llm.with_structured_output(DecisionMakingOutput)
agent_llm = base_llm.bind_tools(tools)
judge_llm = base_llm.with_structured_output(JudgeOutput)


# Decision making node
def decision_making_node(state: AgentState):
    """入口：判断是否需要检索证据。"""
    system_prompt = SystemMessage(content=decision_making_prompt)
    response: DecisionMakingOutput = decision_making_llm.invoke([system_prompt] + state["messages"])
    output = {"requires_research": response.requires_research}
    if (not response.requires_research) and response.answer:
        output["messages"] = [AIMessage(content=response.answer)]
    return output


# Router
def router(state: AgentState):
    if state["requires_research"]:
        return "planning"
    return "end"


# Planning node
def planning_node(state: AgentState):
    system_prompt = SystemMessage(content=planning_prompt.format(tools=format_tools_description(tools)))
    response = base_llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


# Tools node
def tools_node(state: AgentState):
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = tools_dict[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result, ensure_ascii=False),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}


# Agent node
def agent_node(state: AgentState):
    system_prompt = SystemMessage(content=agent_prompt)
    response = agent_llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


# Continue?
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "continue"
    return "end"


# Judge node
def judge_node(state: AgentState):
    num_feedback_requests = state.get("num_feedback_requests", 0)
    if num_feedback_requests >= 2:
        return {"is_good_answer": True}

    system_prompt = SystemMessage(content=judge_prompt)
    response: JudgeOutput = judge_llm.invoke([system_prompt] + state["messages"])
    output = {
        "is_good_answer": response.is_good_answer,
        "num_feedback_requests": num_feedback_requests + 1,
    }
    if response.feedback:
        output["messages"] = [AIMessage(content=response.feedback)]
    return output


# Final answer router
def final_answer_router(state: AgentState):
    if state["is_good_answer"]:
        return "end"
    return "planning"

## 工作流定义

这一节用 LangGraph 把各节点连成完整工作流。

In [ ]:
# Initialize the StateGraph
workflow = StateGraph(AgentState)

# Add nodes to the graph
workflow.add_node("decision_making", decision_making_node)
workflow.add_node("planning", planning_node)
workflow.add_node("tools", tools_node)
workflow.add_node("agent", agent_node)
workflow.add_node("judge", judge_node)

# Set the entry point of the graph
workflow.set_entry_point("decision_making")

# Add edges between nodes
workflow.add_conditional_edges(
    "decision_making",
    router,
    {
        "planning": "planning",
        "end": END,
    }
)
workflow.add_edge("planning", "agent")
workflow.add_edge("tools", "agent")
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",
        "end": "judge",
    },
)
workflow.add_conditional_edges(
    "judge",
    final_answer_router,
    {
        "planning": "planning",
        "end": END,
    }
)

# Compile the graph
app = workflow.compile()

## 示例：学术研究场景

下面用几条示例输入来测试工作流。它们主要用于观察系统在以下方面的行为：
- 能否完成典型的科研调研任务
- 能否在限定条件下检索与筛选论文
- 能否覆盖多个研究方向的查询
- 能否在回答中引用具体证据来支撑结论

In [ ]:
test_inputs = [
    "下载并总结这篇论文的主要结论：https://pmc.ncbi.nlm.nih.gov/articles/PMC11379842/pdf/11671_2024_Article_4070.pdf",

    "你能帮我找 8 篇关于量子机器学习（quantum machine learning）的论文吗？",

    """寻找 2023-2024 年关于 CRISPR 用于治疗遗传疾病的近期论文，重点关注临床试验与安全协议。""",

    """寻找并分析 2023-2024 年 transformer 架构在蛋白质折叠预测中的应用论文，重点关注有实验验证的新颖结构改动。""",
]

outputs = []
for test_input in test_inputs:
    final_answer = await print_stream(app, test_input)
    outputs.append(final_answer.content if final_answer else "")

## 展示结果

这一节把上面几条查询的输入与输出更紧凑地展示出来。

In [ ]:
for input, output in zip(test_inputs, outputs):
    display(Markdown(f"## 输入\n\n{input}\n\n"))
    display(Markdown(f"## 输出\n\n{output}\n\n"))

---

## 对比分析

这里给出一个对比思路：把同一个研究查询（例如“找 8 篇量子机器学习论文”）同时交给不同系统，比较它们在速度、元数据完整性、引用严谨性与输出结构上的差异。

#### 测试设置

- 查询："Find 8 papers on quantum machine learning"
- 次数：多次重复运行以观察稳定性
- 时间：2024 年初
- 指标：响应时间、元数据质量、结果结构

#### 关键观察

不同系统往往体现出“速度 vs 深度”的权衡：更严谨的学术检索与校验通常更慢，但更容易得到可追溯的证据链。

### Microsoft Copilot 示例

![image](https://i.ibb.co/y4Zf4Pc/Screenshot-2024-11-17-at-21-40-21.png)

### Perplexity AI 示例

![image](https://i.ibb.co/n1rr7kW/Screenshot-2024-11-17-at-21-40-42.png)

### 指标对比

![image](https://i.ibb.co/5KbTmFq/Screenshot-2024-11-17-at-22-03-43.png)


上面的图示展示了三类系统在速度与学术严谨性上的差异。你可以把它当作一种选型参考：

- 如果目标是快速浏览与初步探索，速度往往更重要
- 如果目标是严谨的学术调研与可追溯的证据链，通常需要更多校验步骤与更完整的元数据

---



## 局限性

1) 技术层面
- 论文访问可能受 API 限流影响
- 大体积 PDF 的下载与抽取耗时较长
- 仅覆盖公开可访问的论文

2) 能力层面
- 不包含论文图片/图表的深入解析
- 超长文档仍受上下文窗口限制
- 不擅长复杂的数学推导
- 非英文论文可能受到语言能力限制

## 潜在改进

1) 技术改进
- 并行处理多篇论文
- 对高频访问结果做缓存
- 接入更多学术数据源
- 支持批处理与离线任务

2) 功能改进
- 抽取图表/表格并结构化
- 跨论文交叉引用与一致性校验
- 引用网络与研究脉络分析
- 针对特定学科加入更强的规则校验

3) 体验改进
- 更交互式的反馈机制
- 进度与阶段可视化
- 可配置的评审标准
- 导出研究摘要与笔记

## 适用场景

- 学术研究：文献检索、综述与证据整理
- 产业研究：技术文档与专利的快速调研
- 教育学习：引导式学习与资料整理

---

## 总结

这个实现展示了如何用 state-driven 的方式组织科研论文调研：用 LangGraph 编排“决策→规划→工具执行→回答→评审”的流程，并把证据与结论通过 `messages` 串起来。通过在工作流中加入质量评审与必要的迭代改进，可以在自动化处理的同时尽量保持研究严谨性。